# Phase 3B on the SURF A10 VM

Use the frozen selection and original videos copied from Colab. This notebook does not re-screen cases or run Part 1/Phase 1. All outputs and model caches use `/data/yuxuanstorage`. Start with the CPU plan, then VM preflight on both GPUs. Inspect the preflight and relocation-control results before explicitly selecting `full`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

STORAGE_ROOT = Path('/data/yuxuanstorage')
PROJECT_ROOT = STORAGE_ROOT / 'vlm-event-boundary'
PHASE3B_VM_SELECTION = STORAGE_ROOT / 'vlm_phase3b/analysis/selection_v2_controls'
PHASE3B_VM_POOL = STORAGE_ROOT / 'vlm_phase3b/rescue_pool'
PHASE3B_VM_OUTPUT_ROOT = STORAGE_ROOT / 'vlm_phase3b/a10_runs/run_v1'
PHASE3B_VM_GPUS = '0,1'
PHASE3B_VM_STAGE = 'preflight'  # Set to full only after reviewing the VM preflight.

if not (PROJECT_ROOT / 'scripts/run_phase3b_vm.py').is_file():
    raise FileNotFoundError('Set PROJECT_ROOT to the repository on persistent storage.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
vm_common = [
    '--selection_dir', str(PHASE3B_VM_SELECTION),
    '--rescue_pool_root', str(PHASE3B_VM_POOL),
    '--output_root', str(PHASE3B_VM_OUTPUT_ROOT),
    '--storage_root', str(STORAGE_ROOT), '--project_root', str(PROJECT_ROOT),
    '--gpus', PHASE3B_VM_GPUS,
]


## CPU Plan

Checks all frozen video paths and hashes before loading a model. The `1001_phase3B_output.zip` archive contains selection and mapping files but no videos: copy the Drive rescue pool and the existing L5 full videos separately.


In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/run_phase3b_vm.py', '--stage', 'plan', *vm_common], check=True)


## Run or Resume

The two workers receive disjoint five-case shards. Each worker sees only its assigned GPU. Both A10s must pass the two-case preflight, followed by the matched relocation control, before `full` is allowed. Re-running the same configuration resumes its saved checkpoints. Logs and per-GPU progress are saved in the persistent output directory. Do not reuse the A100 checkpoint directory.


In [ ]:
if PHASE3B_VM_STAGE not in {'preflight', 'full', 'analyze'}:
    raise ValueError('Choose preflight, full, or analyze.')
subprocess.run([sys.executable, '-u', 'scripts/run_phase3b_vm.py', '--stage', PHASE3B_VM_STAGE, *vm_common], check=True)


## Back Up After Every Run

This creates timestamped ZIP parts with activation tensors, run artifacts, and the frozen source videos, plus SHA-256 checksums. It also works after a failed/interrupted run once workers have stopped. Download **all** parts, `backup_manifest.json`, and `SHA256SUMS` through the VM file browser or scp/rsync. Packaging on VM is not yet a local backup. If persistent disk space cannot hold another copy, transfer the run directory and frozen videos directly instead; do not delete checkpoints.


In [ ]:
from scripts.backup_phase3b import create_backup

backup_bundle = create_backup(PHASE3B_VM_OUTPUT_ROOT, STORAGE_ROOT / 'backups', reports_only=False)
print('Download every file in:', backup_bundle)
print('Files:', [path.name for path in sorted(backup_bundle.iterdir())])
print('On your local computer, verify: python scripts/backup_phase3b.py --verify <downloaded-directory>')
print('Keep the VM checkpoints until the local backup has been verified.')
